# Noise Schedule — 面试版

**难度：** Easy

**面试要点：** 手写 cumprod + sigmoid，不用内置函数

In [ ]:
import torch
import math

In [ ]:
# ✅ INTERVIEW

def noise_schedule(num_timesteps, schedule_type='cosine'):
    T = num_timesteps
    t = torch.arange(T, dtype=torch.float32)

    if schedule_type == 'linear':
        beta_start, beta_end = 1e-4, 0.02
        betas = beta_start + (beta_end - beta_start) * t / (T - 1)

        # ---- 手写 cumprod ----
        # cumprod(1-β) = [(1-β_0), (1-β_0)(1-β_1), ...]
        # 面试时可以手动实现:
        alphas = 1 - betas
        alpha_bar = torch.ones(T)
        alpha_bar[0] = alphas[0]
        for i in range(1, T):
            alpha_bar[i] = alpha_bar[i-1] * alphas[i]

    elif schedule_type == 'cosine':
        s = 0.008
        f = torch.cos(((t / T + s) / (1 + s)) * (math.pi / 2)) ** 2
        f0 = math.cos((s / (1 + s)) * (math.pi / 2)) ** 2
        alpha_bar = (f / f0).clamp(0.0001, 0.9999)

    elif schedule_type == 'sigmoid':
        beta_start, beta_end = 1e-4, 0.02
        x = t / T * 12 - 6

        # ---- 手写 sigmoid ----
        # sigmoid(x) = 1 / (1 + exp(-x))
        sig = 1.0 / (1.0 + torch.exp(-x))

        betas = beta_start + (beta_end - beta_start) * (sig - sig.min()) / (sig.max() - sig.min())

        # 手写 cumprod
        alphas = 1 - betas
        alpha_bar = torch.ones(T)
        alpha_bar[0] = alphas[0]
        for i in range(1, T):
            alpha_bar[i] = alpha_bar[i-1] * alphas[i]

    else:
        raise ValueError(f'Unknown schedule_type: {schedule_type}')

    return alpha_bar

In [ ]:
T = 1000
for st in ['linear', 'cosine', 'sigmoid']:
    ab = noise_schedule(T, st)
    print(f"{st:>8}: ab[0]={ab[0].item():.4f}, ab[T/2]={ab[T//2].item():.4f}, ab[T-1]={ab[T-1].item():.4f}")

In [ ]:
from torch_judge import check
check("noise_schedule")

## 📝 核心思路总结

1. **手写 cumprod**：逐元素累积乘法，for 循环实现
2. **手写 sigmoid**：`1/(1+exp(-x))`，数值安全（x 在 [-6,6] 范围内）
3. **三种调度**：linear 最简单，cosine 最平滑，sigmoid 介于两者之间